In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import pyreadr
from matplotlib.lines import Line2D
from shapely.geometry import Polygon, Point
from shapely.ops import unary_union
from matplotlib.patches import Polygon as MPLPolygon
import contextily as ctx

pd.options.display.max_colwidth = 100
pd.options.display.max_rows = 10
pd.options.display.max_columns = 30

In [ ]:
### data load 

# load and process counties
counties = gpd.read_file('~/Desktop/Desktop/epidemiology_PhD/01_data/clean/us_cnty_boundaries.geojson')
# define county groups
county_groups = {
    # group 1: LA County
    '06037': 'LA',
    
    # group 2: northwest counties
    '06111': 'Northwest',  # Ventura
    '06083': 'Northwest',  # Santa Barbara
    '06079': 'Northwest',  # San Luis Obispo
    '06029': 'Northwest',  # Kern
    
    # group 3: southeast counties
    '06059': 'Southeast',  # Orange
    '06073': 'Southeast',  # San Diego
    '06025': 'Southeast',  # Imperial
    '06065': 'Southeast',  # Riverside
    '06071': 'Southeast'   # San Bernardino
}
# list of counties to keep
county_fips_to_keep = list(county_groups.keys())
# filter counties dataset
counties_subset = counties[counties['fips'].isin(county_fips_to_keep)]

# load smoke datasets
epa_stations = gpd.read_file('../01_data/01_raw/childs_pm/epa_station_locations/epa_station_locations.shp')
station_smoke = pyreadr.read_r('../01_data/01_raw/childs_pm/station_smokePM_2025_01.rds')[None]
station_smoke_gdf = epa_stations.merge(station_smoke, left_on='stn_id', right_on='id', how='left')
station_smoke_gdf = gpd.GeoDataFrame(station_smoke_gdf, crs=station_smoke_gdf.crs)

counties_subset = counties_subset.to_crs(epa_stations.crs)  # convert to same CRS as smoke
station_smoke_gdf = gpd.sjoin(station_smoke_gdf, counties_subset[['fips', 'geometry']], 
                                 how='inner', predicate='intersects')

# for now, let's use the average smoke pm for each station during the 1 week period 
station_averages = station_smoke_gdf[station_smoke_gdf['date'] > '2025-01-06' and station_smoke_gdf < '2025-01-14']
station_averages = station_averages.groupby('stn_id').agg({
    'smokePM': 'mean',
    'geometry': 'first'  # keep the first geometry for each station
}).reset_index()

# load fires
fires = gpd.read_file('../01_data/01_raw/data_2025_01_17.geojson').to_crs(epsg=2229)
fires["poly_DateCurrent"] = fires["poly_DateCurrent"].dt.tz_convert('US/Pacific')
fires = fires[fires['poly_DateCurrent'] > '2025-01-06']
fires["poly_DateCurrent"] = fires["poly_DateCurrent"].dt.date
# fires = fires[['geometry']]
# fires_union = fires.dissolve()  # dissolve to one multipolygon
# fires_union = fires_union.to_crs(epa_stations.crs)

## for now lets just use the eaton fire for our cone 
fire_eaton = fires[fires['incident_name'].isin(['Eaton', 'EATON'])]
fire_eaton = fire_eaton.to_crs(epa_stations.crs)
fire_eaton = fire_eaton.dissolve()  # dissolve to one multipolygon
fire_eaton['centroid'] = fire_eaton.geometry.centroid
fire_eaton_centroid = fire_eaton.copy()
fire_eaton_centroid['geometry'] = fire_eaton_centroid['centroid']
fire_eaton_centroid

# load census tracts

In [ ]:
station_smoke_gdf['date'] = pd.to_datetime(station_smoke_gdf['date'])
station_averages = station_smoke_gdf[
    (station_smoke_gdf['date'] > '2025-01-06') & 
    (station_smoke_gdf['date'] < '2025-01-14')
]
station_averages = station_averages.groupby('stn_id').agg({
    'smokePM': 'mean',
    'geometry': 'first'  # keep the first geometry for each station
}).reset_index()

# convert to gdf 
station_averages = gpd.GeoDataFrame(station_averages, geometry='geometry', crs=station_smoke_gdf.crs)
station_averages

In [ ]:
m = fire_eaton.explore(name="Fire perimeter", color='red', alpha=0.7)
fire_eaton_centroid.explore(m=m, name="Centroid", color='blue', marker_kwds={'radius': 8})
m

In [ ]:
# ========================================
# STEP 1: params
# ========================================
# fire center -- replace depending on if we use other fires?
# using the centroid of the eaton fire for now
fire_center = fire_eaton_centroid

# pm threshold
pm_threshold = 5

# distance to end of cone
cone_length = 100000  # 100km

# crs
crs_wm = 3857  # web mercator

In [ ]:
# ========================================
# STEP 2: convert to projected CRS
# ========================================
# convert station data to Web Mercator for distance calculations
station_averages_mercator = station_averages.to_crs(epsg=crs_wm)

# convert fire center to same CRS
fire_point_gdf = gpd.GeoDataFrame([1], geometry=[fire_center.geometry.iloc[0]], crs=station_averages.crs)
fire_point_mercator = fire_point_gdf.to_crs(epsg=crs_wm).geometry.iloc[0] # TODO: SOMETHING IS WRONG WITH THIS! 

In [ ]:
# PM distribution - subset to over 0
pm_data = station_averages_mercator['smokePM']
pm_subset = pm_data[pm_data > 0]

plt.figure(figsize=(10, 4))
plt.hist(pm_subset, bins=50, alpha=0.7)
plt.axvline(pm_threshold, color='red', linestyle='--', label=f'Threshold: {pm_threshold}')
plt.xlabel('PM2.5')
plt.ylabel('# of stations')
plt.title(f'distribution of PM2.5 values (over 0) - {len(pm_subset)} of {len(pm_data)} stations shown')
plt.legend()
plt.show()

In [ ]:
# ========================================
# STEP 3: Separate stations by PM threshold
# ========================================
high_pm_stations = station_averages_mercator[station_averages_mercator['smokePM'] > pm_threshold]
low_pm_stations = station_averages_mercator[station_averages_mercator['smokePM'] <= pm_threshold]

print(f"\nStation groups:")
print(f"High PM stations (>{pm_threshold}): {len(high_pm_stations)}")
print(f"Low PM stations (<={pm_threshold}): {len(low_pm_stations)}")

# Check the PM values
if len(high_pm_stations) > 0:
    print(f"High PM range: {high_pm_stations['smokePM'].min():.1f} to {high_pm_stations['smokePM'].max():.1f}")
if len(low_pm_stations) > 0:
    print(f"Low PM range: {low_pm_stations['smokePM'].min():.1f} to {low_pm_stations['smokePM'].max():.1f}")

# plot the high vs low stations
fig, ax = plt.subplots(figsize=(10, 10))
station_averages_mercator.plot(ax=ax, color='lightgray', markersize=5, label='All Stations')
high_pm_stations.plot(ax=ax, color='red', markersize=10, label='high PM')
low_pm_stations.plot(ax=ax, color='orange', markersize=10, label='low PM')
# add fire perimeter and centroid
fire_eaton.to_crs(crs_wm).plot(ax=ax, color='blue', alpha=0.5, edgecolor='black', label='fire perimeter')
ctx.add_basemap(ax, crs=station_averages_mercator.crs.to_string(), source=ctx.providers.CartoDB.Positron)
ax.legend()
plt.show()


In [ ]:
# ========================================
# STEP 4: Find anchor stations for polygons
# ========================================

def find_nearest_stations(stations, fire_point, n_stations=2):
    """Find the n nearest stations to fire center"""
    distances = []
    for idx, row in stations.iterrows():
        dist = fire_point.distance(row.geometry)
        distances.append((dist, idx, row))
    
    # Sort by distance and take nearest
    distances.sort(key=lambda x: x[0])
    return distances[:n_stations]

def find_furthest_stations(stations, fire_point, n_stations=2):
    """Find the n furthest stations from fire center"""
    distances = []
    for idx, row in stations.iterrows():
        dist = fire_point.distance(row.geometry)
        distances.append((dist, idx, row))
    
    # Sort by distance and take furthest
    distances.sort(key=lambda x: x[0], reverse=True)
    return distances[:n_stations]

# Find anchor stations
nearest_low_pm = find_nearest_stations(low_pm_stations, fire_point_mercator, 2)
furthest_high_pm = find_furthest_stations(high_pm_stations, fire_point_mercator, 2)

print("\nAnchor stations:")
if nearest_low_pm:
    print("Nearest low PM stations:")
    for dist, idx, row in nearest_low_pm:
        print(f"  Distance: {dist/1000:.1f} km, PM: {row['smokePM']:.1f}")

if furthest_high_pm:
    print("Furthest high PM stations:")
    for dist, idx, row in furthest_high_pm:
        print(f"  Distance: {dist/1000:.1f} km, PM: {row['smokePM']:.1f}")

fig, ax = plt.subplots(figsize=(10, 10))

# Plot all stations first
station_averages_mercator.plot(ax=ax, color='lightgray', markersize=5, label='All Stations')
high_pm_stations.plot(ax=ax, color='red', markersize=10, label='high PM')
low_pm_stations.plot(ax=ax, color='orange', markersize=10, label='low PM')

# Highlight selected anchor stations in yellow
if nearest_low_pm:
   for dist, idx, row in nearest_low_pm:
       ax.scatter(row.geometry.x, row.geometry.y, c='yellow', s=150, marker='s', 
                 zorder=10, alpha = 0.5)

if furthest_high_pm:
   for dist, idx, row in furthest_high_pm:
       ax.scatter(row.geometry.x, row.geometry.y, c='yellow', s=150, marker='s', 
                 zorder=10, alpha = 0.5)

# Add fire perimeter and centroid
fire_eaton.to_crs(crs_wm).plot(ax=ax, color='blue', alpha=0.5, edgecolor='black', label='fire perimeter')
ax.scatter(fire_point_mercator.x, fire_point_mercator.y, c='blue', s=200, marker='*', 
          edgecolor='black', linewidth=2, zorder=15, label='fire center')

# Add custom legend entry for selected stations
from matplotlib.lines import Line2D
legend_elements = ax.get_legend_handles_labels()[0] + [
   Line2D([0], [0], marker='s', color='w', markerfacecolor='yellow', 
          markersize=10, markeredgecolor='black', label='Selected anchor stations')
]
legend_labels = ax.get_legend_handles_labels()[1] + ['Selected anchor stations']

ctx.add_basemap(ax, crs=station_averages_mercator.crs.to_string(), source=ctx.providers.CartoDB.Positron)
ax.legend(legend_elements, legend_labels)
plt.show()

In [ ]:

# ========================================
# STEP 5: Create polygons
# ========================================

def create_elbow_polygon(fire_point, anchor_stations, cone_length, default_angle=45):
    """
    Create polygon going through anchor stations
    
    Parameters:
    - fire_point: Point geometry of fire center
    - anchor_stations: list of (distance, idx, row) tuples
    - cone_length: how far to extend into ocean
    - default_angle: angle in degrees for lines without anchor stations
    """
    fire_x, fire_y = fire_point.x, fire_point.y
    polygon_points = [(fire_x, fire_y)]  # Start at fire center
    
    if len(anchor_stations) >= 2:
        # Use actual anchor stations
        station1 = anchor_stations[0][2]  # row from first station
        station2 = anchor_stations[1][2]  # row from second station
        
        # Add the anchor stations to polygon
        polygon_points.append((station1.geometry.x, station1.geometry.y))
        polygon_points.append((station2.geometry.x, station2.geometry.y))
        
        # Calculate angles from fire center to each station
        angle1 = np.arctan2(station1.geometry.y - fire_y, station1.geometry.x - fire_x)
        angle2 = np.arctan2(station2.geometry.y - fire_y, station2.geometry.x - fire_x)
        
        # Extend lines from each station outward
        # Extension from station 1
        ext1_x = station1.geometry.x + cone_length * np.cos(angle1)
        ext1_y = station1.geometry.y + cone_length * np.sin(angle1)
        polygon_points.append((ext1_x, ext1_y))
        
        # Extension from station 2  
        ext2_x = station2.geometry.x + cone_length * np.cos(angle2)
        ext2_y = station2.geometry.y + cone_length * np.sin(angle2)
        polygon_points.append((ext2_x, ext2_y))
        
    elif len(anchor_stations) == 1:
        # Use one anchor station and default angle for other line
        station = anchor_stations[0][2]
        polygon_points.append((station.geometry.x, station.geometry.y))
        
        # Calculate angle to station
        station_angle = np.arctan2(station.geometry.y - fire_y, station.geometry.x - fire_x)
        
        # Extend from station
        ext1_x = station.geometry.x + cone_length * np.cos(station_angle)
        ext1_y = station.geometry.y + cone_length * np.sin(station_angle)
        polygon_points.append((ext1_x, ext1_y))
        
        # Create second line using default angle
        default_angle_rad = np.radians(default_angle)
        ext2_x = fire_x + cone_length * np.cos(default_angle_rad)
        ext2_y = fire_y + cone_length * np.sin(default_angle_rad)
        polygon_points.append((ext2_x, ext2_y))
        
    else:
        # No anchor stations - use default angles
        angle1_rad = np.radians(default_angle)
        angle2_rad = np.radians(default_angle + 30)  # 30 degree spread
        
        ext1_x = fire_x + cone_length * np.cos(angle1_rad)
        ext1_y = fire_y + cone_length * np.sin(angle1_rad)
        polygon_points.append((ext1_x, ext1_y))
        
        ext2_x = fire_x + cone_length * np.cos(angle2_rad)
        ext2_y = fire_y + cone_length * np.sin(angle2_rad)
        polygon_points.append((ext2_x, ext2_y))
    
    # Close polygon back to fire center
    polygon_points.append((fire_x, fire_y))
    
    return Polygon(polygon_points)

# Create the polygons
red_polygon = None
orange_polygon = None

# Red polygon: through nearest low PM stations
if nearest_low_pm:
    red_polygon = create_elbow_polygon(
        fire_point_mercator, 
        nearest_low_pm, 
        cone_length, 
        default_angle=45  # you can adjust this
    )
    print(f"\nRed polygon created with area: {red_polygon.area/1e6:.1f} km²")

# Orange polygon: through furthest high PM stations  
if furthest_high_pm:
    orange_polygon = create_elbow_polygon(
        fire_point_mercator,
        furthest_high_pm,
        cone_length,
        default_angle=135  # you can adjust this
    )
    print(f"Orange polygon created with area: {orange_polygon.area/1e6:.1f} km²")

# ========================================
# STEP 6: Visualize the elbow polygons
# ========================================

fig, ax = plt.subplots(figsize=(12, 10))

# Plot polygons first (behind stations)
if orange_polygon:
    from matplotlib.patches import Polygon as MPLPolygon
    orange_patch = MPLPolygon(
        list(orange_polygon.exterior.coords),
        facecolor='orange', alpha=0.3, edgecolor='orange', linewidth=2,
        label='Orange polygon (furthest high PM)'
    )
    ax.add_patch(orange_patch)

if red_polygon:
    red_patch = MPLPolygon(
        list(red_polygon.exterior.coords),
        facecolor='red', alpha=0.3, edgecolor='red', linewidth=2,
        label='Red polygon (nearest low PM)'
    )
    ax.add_patch(red_patch)

# Plot all stations
station_averages_mercator.plot(ax=ax, color='lightgray', markersize=5, alpha=0.6)
high_pm_stations.plot(ax=ax, color='red', markersize=8, alpha=0.8, label='High PM stations')
low_pm_stations.plot(ax=ax, color='orange', markersize=8, alpha=0.8, label='Low PM stations')

# Highlight anchor stations
if nearest_low_pm:
    for dist, idx, row in nearest_low_pm:
        ax.scatter(row.geometry.x, row.geometry.y, c='red', s=150, marker='s', 
                  edgecolor='black', linewidth=2, zorder=10)

if furthest_high_pm:
    for dist, idx, row in furthest_high_pm:
        ax.scatter(row.geometry.x, row.geometry.y, c='orange', s=150, marker='s', 
                  edgecolor='black', linewidth=2, zorder=10)

# Plot fire center and perimeter
fire_eaton.to_crs(crs_wm).plot(ax=ax, color='blue', alpha=0.5, edgecolor='black')
ax.scatter(fire_point_mercator.x, fire_point_mercator.y, c='blue', s=200, marker='*', 
           edgecolor='black', linewidth=2, zorder=15, label='Fire center')

# Add basemap
ctx.add_basemap(ax, crs=station_averages_mercator.crs.to_string(), source=ctx.providers.CartoDB.Positron)

ax.legend(loc='upper left')
ax.set_title('Elbow-Style Polygons Through Anchor Stations')
plt.show()

print("\n" + "="*50)
print("ELBOW POLYGONS CREATED!")
print("="*50)
print("Variables created:")
print("- red_polygon: Goes through nearest low PM stations")
print("- orange_polygon: Goes through furthest high PM stations")
print("- nearest_low_pm: List of anchor stations for red polygon")
print("- furthest_high_pm: List of anchor stations for orange polygon")
print("\nYou can adjust the default_angle parameters to change line directions")
print("when there aren't enough anchor stations available.")